<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [6]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [8]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Experiment 1: Train Logistic Regression Classifier

In [9]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



### Experiment 2: Train Random Forest Classifier

In [10]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       270
           1       0.95      0.70      0.81        30

    accuracy                           0.97       300
   macro avg       0.96      0.85      0.89       300
weighted avg       0.97      0.97      0.96       300



### Experiment 3: Train XGBoost

In [11]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [12]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [13]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



<h2 align="center" style="color:blue">Track Experiments Using MLFlow</h2>

In [14]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": "liblinear"},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": "logloss"},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": "logloss"},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [15]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model = model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)
    
reports[1]

{'0': {'precision': 0.9676258992805755,
  'recall': 0.9962962962962963,
  'f1-score': 0.9817518248175182,
  'support': 270.0},
 '1': {'precision': 0.9545454545454546,
  'recall': 0.7,
  'f1-score': 0.8076923076923077,
  'support': 30.0},
 'accuracy': 0.9666666666666667,
 'macro avg': {'precision': 0.961085676913015,
  'recall': 0.8481481481481481,
  'f1-score': 0.8947220662549129,
  'support': 300.0},
 'weighted avg': {'precision': 0.9663178548070633,
  'recall': 0.9666666666666667,
  'f1-score': 0.9643458731049971,
  'support': 300.0}}

In [16]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [21]:
# Initialize MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Anomaly Detection with Params")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)  
        mlflow.log_metric("accuracy", report['accuracy'])
        mlflow.log_metric("recall_0", report['0']['recall'])
        mlflow.log_metric("recall_1", report['1']['recall'])
        mlflow.log_metric("f1_score_macro", report["macro avg"]["f1-score"])
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, artifact_path=model_name)
        else:
            mlflow.sklearn.log_model(model, artifact_path=model_name)
    




2026/01/28 13:16:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 13:16:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/3/runs/19bc9c29afd14a7f88e0ce26a32c5a0e
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/01/28 13:16:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 13:16:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/3/runs/63b38995fe7749ff961c7eb9184eed8b
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/01/28 13:16:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 13:16:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/3/runs/589d730d6afc4b2ebc05a9292b4f7d66
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/01/28 13:16:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 13:16:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/3/runs/59af5de99f194ef0b70378c65ac5dc8c
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Register the model


In [25]:
model_name = "XGBClassifier With SMOTE"
run_id = input("Enter the run ID to register the model: ")
model_uri = f"runs:/{run_id}/{model_name}"
result = mlflow.register_model(model_uri, model_name)

Registered model 'XGBClassifier With SMOTE' already exists. Creating a new version of this model...
2026/01/28 13:24:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 59af5de99f194ef0b70378c65ac5dc8c has no artifacts at artifact path 'XGBClassifier With SMOTE', registering model based on models:/m-ae188a95cb1a479ebc8affae9f448d15 instead
2026/01/28 13:24:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier With SMOTE, version 2
Created version '2' of model 'XGBClassifier With SMOTE'.


### Load the model


In [26]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [27]:
dev_model = f"models:/{model_name}@challenger"
prod_model = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model, dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '2' of model 'XGBClassifier With SMOTE' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1769606888005, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1769606888005, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='59af5de99f194ef0b70378c65ac5dc8c', run_link='', source='models:/XGBClassifier With SMOTE/2', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [28]:
model_version = 1
model_uri = f"models:/{prod_model}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300

